In [ ]:
# 1. مسح ملف النموذج المؤقت الذي يسبب المشكلة
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft/Phi*

# 2. تحديث مكتبة transformers لإصدار يحتوي على seen_tokens
!pip install -U transformers>=4.44.0

# 3. تحديث accelerate لدعم الـ GPU بشكل أفضل
!pip install -U accelerate

In [ ]:
!pip install -q transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
SAVE_PATH = "/content/drive/MyDrive/edu_agent_model"
READY_FLAG = os.path.join(SAVE_PATH, "ready.txt")

Mounted at /content/drive


In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU متاح: {gpu_name}")
    print(f"📦 VRAM: {vram:.1f} GB")
else:
    print("⚠️ ما في GPU — رح يشتغل على CPU بس رح يكون بطيء")

✅ GPU متاح: Tesla T4
📦 VRAM: 15.6 GB


In [ ]:
# ═══════════════════════════════════════
#  الإعدادات — نعدّلها من هنا فقط
# ═══════════════════════════════════════
from transformers import pipeline

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

QUANTIZATION_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,
}

GENERATION_CONFIG = {
    "max_new_tokens": 512,  # عدد الكلمات المسموح بتوليدها
    "temperature": 0.7,      # للتنويع في الردود
    "top_p": 0.9,
    "do_sample": False,
}



In [ ]:
import os

if os.path.exists(READY_FLAG):
    print("⚡ تحميل النموذج من Drive...")
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model = AutoModelForCausalLM.from_pretrained(SAVE_PATH, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
    print("✅ تم التحميل من Drive")


else:
    print("📥 أول مرة — جاري التنزيل والضغط...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch

    bnb_config = QUANTIZATION_CONFIG

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("💾 جاري حفظ النموذج على Drive...")
    os.makedirs(SAVE_PATH, exist_ok=True)
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)

    with open(READY_FLAG, "w") as f:
        f.write("ok")

    print(f"✅ تم الحفظ على Drive")

⚡ تحميل النموذج من Drive...


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ تم التحميل من Drive


In [ ]:
llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    return_full_text=False,  # نرجع بس النص الجديد، مش البرومبت كامل
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def generate_response(messages: list) -> str:
    """توليد رد — نمرر لها قائمة رسائل"""
    # تحويل الرسائل للصيغة الرسمية للنموذج تلقائياً
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        **GENERATION_CONFIG,
        pad_token_id=tokenizer.pad_token_id,
    )

    # استخراج الرد فقط بدون الـ prompt
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

# ─── اختبار سريع ───
#msgs = [{"role": "user", "content": "اشرح لي مفهوم الذكاء الاصطناعي بجملتين فقط."}]
#print(f"🤖: {generate_response(msgs)}")

### 🛠️ منهجية بناء الإيجنت وتصميم النظام

#### 1. لماذا الرد المهيكل (JSON)

؟
* **الفهم البرمجي:**

الإيجنت يجب أن يعيد ردوداً بصيغة JSON لتمكين النظام من قراءة القرارات آلياً.
* **تجنب الكوابيس:**

 الاعتماد على معالجة النصوص (Parsing) المعقدة للغة البشرية هو "كابوس" تقني يزيد من احتمالية الخطأ.

#### 2. إدارة الحالة (State Management)


* **الحماية والتنظيم:**

 يتم عزل الـ State في كلاس (Class) منفصل لضمان الخصوصية والتنظيم.
* **التتبع والحفظ:**

يسهل هذا التصميم عملية مراقبة حالة الإيجنت في كل لحظة، أو طباعتها، أو حتى حفظها للرجوع إليها لاحقاً.

#### 3. حلقة التحكم (Control Loop)


* **البساطة أولاً:**

 نستخدم حلقة `while` مع شرطين (المهمة لم تحل + عدد الخطوات أقل من 10).
* **المرونة:**

حلقة `while` أفضل من `for` لأنها تسمح بالإيقاف الفوري في حالات معينة، مثل رفض الطلب من قبل الإيجنت أو الوصول لحل نهائي قبل انتهاء العداد.

In [ ]:
SYSTEM_PROMPT = """You are a strict Triage Agent. Your ONLY job is to analyze user requests and choose EXACTLY ONE of the following 3 actions. Do NOT answer the question directly, and do NOT write code.

ACTIONS & CONDITIONS:
1. [CREATE_TICKET]: USE IF the request is clear, specific, and requires a solution. (Briefly summarize the clear goal in plain text).
2. [ASK_MORE_INFO]: USE IF the request is vague, ambiguous, or lacks context. (Ask naturally what is missing).
3. [REJECT_REQUEST]: USE IF the request violates safety, asks to harm the server, or bypass security. (State the rejection reason politely but firmly).

STATE AWARENESS:
Check the current state. If it is "waiting_for_info", evaluate the new input to see if it's now clear enough for [CREATE_TICKET] or still needs [ASK_MORE_INFO].

OUTPUT FORMAT:
You MUST start your response EXACTLY with one of the three tags above, followed by a space, then write your plain text message.
Example: [ASK_MORE_INFO] ما هو نوع السيرفر الذي تقصده؟
"""

### 🎯 فلسفة التصميم التقني للإيجنت

#### 1. لماذا التعابير النمطية (Regex) بدلاً من JSON؟


* **السرعة والخفة:** تم اختيار Regex لأنها توفر أداءً أسرع في استخلاص القرارات.


* **تبسيط المعالجة:** التعابير النمطية قادرة على
التقاط "الأكشن" المطلوب من النص العادي مباشرة دون الحاجة لعمليات Parsing معقدة قد تفشل إذا أخطأ النموذج في تنسيق JSON.


#### 2. أهمية التاجات في بداية النص (`^`)


* **دقة التوجيه:** وضع التاج (مثل `[CREATE_TICKET]`) في بداية الرد يضمن عدم خلط الكود بين "قرار الإيجنت" وأي كلمات مشابهة قد ترد صدفة داخل نص الإجابة.


* **موثوقية الاستخراج:** نضمن أن الكود يقرأ أول كلمة كأمر تنفيذي فقط.

#### 3. تنظيف النص باستخدام `re.sub`


* **فصل المسؤوليات:** نستخدم `re.sub` لفصل "قرار
الإيجنت" (الذي يحتاجه النظام برمجياً) عن "رسالة الإيجنت" (التي تظهر للمستخدم).


* **تجربة مستخدم نظيفة:** ينتج عن هذه العملية متغير `clean_text` الذي يعرض للمستخدم نصاً خالصاً ومهنياً بدون أي أكواد أو تاجات تقنية مشوشة.

# **STATE MANAGEMENT**

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List
import re

# ═══════════════════════════════════════
#  3. إدارة الحالة (State Management)
# ═══════════════════════════════════════
@dataclass
class AgentState:
    user_request: str = ""
    is_resolved: bool = False
    step: int = 0
    max_steps: int = 10
    retries: int = 0
    max_retries: int = 3
    last_action: Optional[str] = None
    history: List[dict] = field(default_factory=list)
    ticket_answer: Optional[str] = None
    rejection_reason: Optional[str] = None

    def can_proceed(self) -> bool:
        return self.step < self.max_steps and not self.is_resolved and self.retries < self.max_retries

    def record(self, action: str, response: str):
        self.last_action = action
        self.history.append({"action": action, "response": response})


# ═══════════════════════════════════════
#  4. الوظائف المساعدة
# ═══════════════════════════════════════
def generate_response(messages):
    output = llm_pipe(messages, temperature=0.1, do_sample=False, return_full_text=False)
    return output[0]['generated_text'].strip()

def build_user_prompt(user_input: str, state: AgentState) -> str:
    if state.step == 0 and not state.history:
        state.user_request = user_input
        return f"طلب المستخدم: {user_input}"
    else:
        history_text = "\n".join([f"- [{h['action']}] {h['response']}" for h in state.history])
        return f"""الطلب الأصلي: {state.user_request}
محاولاتك السابقة:
{history_text}
رد المستخدم الجديد الآن: {user_input}"""

def parse_agent_response(text: str) -> dict:
    clean_raw = text.strip()

    # التعديل هنا: أضفنا \[? و \]? عشان الأقواس تصير اختيارية
    pattern = r"(?:\[)?(CREATE_TICKET|ASK_MORE_INFORMATION|REJECT_REQUEST)(?:\])?\s*(.*)"
    match = re.search(pattern, clean_raw, re.DOTALL)

    if match:
        action = match.group(1)
        clean_text = match.group(2).strip()
        return {"action": action, "response": clean_text}

    print(f"[DEBUG] رد المودل الخام: {clean_raw}")
    raise ValueError("لم يتم العثور على التاج المطلوب")

###  التعامل مع طبيعة النماذج اللغوية الصغيرة (SLMs)

#### 1. المرونة مقابل الصرامة

* **طبيعة الموديلات:**

 نماذج مثل `Phi-3.5-mini` قد تواجه أحياناً صعوبة في الالتزام التام بترميزات النصوص وعلامات الترقيم (Punctuation) تحت الضغط، ولكن أداءها الدلالي وفهمها للمخاطر يظل ممتازاً.
* **فلسفة الكود:**

 في هذه الحالة، تعديل الكود ليكون أكثر مرونة أهم بكثير من محاولة إجبار الموديل على الصرامة المطلقة.

#### 2. شرح التعبير النمطي `(?:\[)?`


* **الوظيفة:**

 يخبر هذا الجزء من الـ Regex المحرك بالبحث عن القوس المربع المفتوح `[`.
* **المرونة:**

 وجود علامة الاستفهام `?` تجعل وجود القوس اختيارياً؛ أي إذا وجده المحرك فهذا جيد، وإذا لم يجده يستمر في العمل دون إطلاق خطأ، مما يضمن استمرارية المعالجة حتى لو أغفل الموديل الأقواس.

#### 3. استخدام `re.search` بدلاً من `re.match`


* **تجاوز الأخطاء الهامشية:**

 `re.match` يشترط وجود النمط في بداية النص تماماً، بينما `re.search` يبحث في النص بالكامل.
* **الهدف:**

 إذا كتب الموديل مسافة زائدة أو حرفاً غريباً بالصدفة قبل التاج، سيتمكن `re.search` من العثور على الأمر المطلوب وتنفيذه، بينما كان `re.match` سيؤدي إلى فشل العملية بالكامل.

# **RUNNER**

In [ ]:
def run_agent(user_request: str):
    state = AgentState()
    current_input = user_request

    print(f"🧑 المستخدم: {current_input}\n")

    while state.can_proceed():
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(current_input, state)}
        ]

        try:
            raw = generate_response(messages)
            parsed = parse_agent_response(raw)
            state.retries = 0

        except Exception as e:
            state.retries += 1
            print(f"⚠️ خطأ في صيغة الرد (محاولة: {state.retries}/{state.max_retries})...\n")
            if state.retries >= state.max_retries:
                state.rejection_reason = "فشل النظام في فهم رد الذكاء الاصطناعي 3 مرات."
                state.is_resolved = True
            continue

        action = parsed["action"]
        response = parsed["response"]

        if action == "CREATE_TICKET":
            state.ticket_answer = response
            state.is_resolved = True
            state.step += 1
            state.record(action, response)

        elif action == "ASK_MORE_INFORMATION":
            state.record(action, response)

        elif action == "REJECT_REQUEST":
            state.rejection_reason = response
            state.is_resolved = True
            state.step += 1
            state.record(action, response)

        else:
            state.retries += 1
            print(f"⚠️ إجراء غير معروف (محاولة: {state.retries}/{state.max_retries})...\n")
            if state.retries >= state.max_retries:
                state.rejection_reason = "صدر إجراء غير معروف من النظام."
                state.is_resolved = True
            continue

        print(f"🤖 الإيجنت [{action}]: {response}\n")

        if state.is_resolved:
            break

        if action == "ASK_MORE_INFORMATION":
            current_input = input("🧑 ردك: ")
            print()

    print("=" * 50)
    if state.ticket_answer:
        print("🎫 تم إنشاء تذكرة بنجاح!")
        print(f"📝 الجواب: {state.ticket_answer}")
    elif state.rejection_reason:
        print("❌ تم إيقاف الطلب (رفض أو خطأ تقني)")
        print(f"🚫 السبب: {state.rejection_reason}")
    else:
        print("⏱️ استنفدت عدد الخطوات الفعلية (10) بدون حل")
    print(f"📊 إجمالي الخطوات الفعلية: {state.step}")
    print("=" * 50)


first_message = input("اكتب رسالتك الأولى: ")
run_agent(first_message)

اكتب رسالتك الأولى: احسب متوسط السنوات الخبرة


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑 المستخدم: احسب متوسط السنوات الخبرة



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[DEBUG] رد المودل الخام: [ASK_MORE_INFO] هل يمكنك إخطار النوع أو المعلومات التي تقدمها لتساعدنا في حساب متوسط السنوات الخبرة المطلوب؟
⚠️ خطأ في صيغة الرد (محاولة: 1/3)...



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[DEBUG] رد المودل الخام: [ASK_MORE_INFO] هل يمكنك إخطار النوع أو المعلومات التي تقدمها لتساعدنا في حساب متوسط السنوات الخبرة المطلوب؟
⚠️ خطأ في صيغة الرد (محاولة: 2/3)...

[DEBUG] رد المودل الخام: [ASK_MORE_INFO] هل يمكنك إخطار النوع أو المعلومات التي تقدمها لتساعدنا في حساب متوسط السنوات الخبرة المطلوب؟
⚠️ خطأ في صيغة الرد (محاولة: 3/3)...

❌ تم إيقاف الطلب (رفض أو خطأ تقني)
🚫 السبب: فشل النظام في فهم رد الذكاء الاصطناعي 3 مرات.
📊 إجمالي الخطوات الفعلية: 0


# **Code-Generation Agent Workflow**

### **منهجية التصميم الأمني والتقني للنظام**

#### **1. طبقة الحماية (Validation Layer)**
لقد قمت ببناء هذه الطبقة باستخدام **Regex Scanning** كخيار استراتيجي.

السبب هو أنها توفر وسيلة خفيفة وسريعة وفعالة جداً لمنع استدعاء مكتبات النظام الحساسة مثل (`OS`, `Subprocess`) وأوامر الحقن البرمجي الخطيرة مثل (`eval`, `exec`).

---

#### **2. نهج الوصف فقط (Schema-Only)**
بدلاً من إرسال البيانات الفعلية للنموذج اللغوي (LLM)، نقوم بإرسال **"وصف الجدول فقط"** (أسماء الأعمدة وأنواع البيانات).

الهدف الأساسي من ذلك هو منع النموذج من حفظ البيانات الحساسة أو التسبب في تسريبها، مما يضمن خصوصية كاملة للمعلومات.

---

#### **3. التنفيذ المعزول (Sandboxed Execution)**
اعتمدت في التنفيذ على تقنية **`redirect_stdout`** مع استخدام **`exec`** في بيئة محصورة تقنياً.

تكمن الفائدة هنا في تشغيل كود مكتبة `Pandas` على الجهاز محلياً بأمان، حيث نقوم فقط بالتقاط المخرجات (Output) النهائية وعرضها لك دون منح الكود صلاحيات أوسع.

---

#### **4. هيكلة الكود وتنظيمه**
حرصت على **فصل الإعدادات (Config) عن المنطق البرمجي (Logic)** بشكل كامل.

هذا التصميم يتيح لك مرونة عالية؛ فإذا أردت إضافة مكتبات ممنوعة جديدة أو تعديل سياسات الأمان، يمكنك القيام بذلك من مكان واحد مركزي دون المساس بجوهر النظام.

# **الخلية 0: إنشاء ملف الإكسل المطلوب وتحميله**

In [ ]:
import pandas as pd
import random

# إنشاء البيانات العشوائية ضمن القيود المحددة
names = ["Ahmed", "Sara", "Omar", "Layla", "Khaled"]
data = {
    "Name": random.sample(names, 5), # ترتيب عشوائي للأسماء
    "salary": [random.randint(1500, 2300) for _ in range(5)],
    "number of years experience": [random.randint(3, 8) for _ in range(5)],
    "age": [random.randint(26, 45) for _ in range(5)]
}

df_mock = pd.DataFrame(data)
file_name = "employees_data.xlsx"

# حفظ الملف
df_mock.to_excel(file_name, index=False)

# تشغيل التحميل التلقائي في كولاب
from google.colab import files
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **الخلية 1: الإعدادات، الحماية، وقراءة البيانات الحقيقية**
السبب: تجميع كل الثوابت (Forbidden Nodes) في مكان واحد يسهل الـ Scaling والتعديل لاحقاً.

In [ ]:
import re
import io
import sys
import ast
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, List

# ══════════════════════════════════════════════════════════
#  الجزء 1: قراءة البيانات + استخراج الـ Schema فقط
# ══════════════════════════════════════════════════════════
FILE_PATH = "employees_data.xlsx"
df_real = pd.read_excel(FILE_PATH)
DATA_SCHEMA = f"أسماء الأعمدة وأنواعها:\n{df_real.dtypes.to_string()}"
MAX_OUTPUT_LINES = 10  # أقصى عدد أسطر مسموح كناتج نهائي
# ══════════════════════════════════════════════════════════
#  الجزء 2: طبقة الأمان — القوائم + AST Validator
# ══════════════════════════════════════════════════════════
FORBIDDEN_MODULES = {
    "os", "sys", "subprocess", "shutil", "socket", "requests",
    "urllib", "http", "ftplib", "smtplib", "ctypes", "pickle",
    "marshal", "importlib", "multiprocessing", "threading",
    "sqlite3", "pathlib", "glob", "__builtins__",
}
FORBIDDEN_CALLS = {
    "eval", "exec", "compile", "__import__", "open", "input",
    "globals", "locals", "vars", "getattr", "setattr", "delattr",
    "breakpoint", "help", "memoryview", "exit", "quit",
}
FORBIDDEN_ATTRS = {
    "__class__", "__base__", "__bases__", "__mro__", "__subclasses__",
    "__globals__", "__builtins__", "__code__", "__closure__",
    "__reduce__", "__reduce_ex__", "__getattribute__", "__init_subclass__",
}
FORBIDDEN_SQL = {
    "SELECT", "DROP", "DELETE", "INSERT", "UPDATE",
    "ALTER", "CREATE", "EXEC", "EXECUTE", "TRUNCATE", "ATTACH", "PRAGMA",
}
SAFE_BUILTIN_CALLS = {
    "print", "round", "len", "sum", "min", "max", "abs",
    "int", "float", "str", "list", "dict", "sorted", "set",
}
FORBIDDEN_NODE_TYPES = (
    ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef,
    ast.ClassDef, ast.For, ast.While, ast.With, ast.Try,
    ast.Global, ast.Nonlocal, ast.Delete, ast.Lambda, ast.AugAssign,
)


def _get_root_name(node):
    while isinstance(node, (ast.Attribute, ast.Subscript, ast.Call)):
        node = node.value if isinstance(node, (ast.Attribute, ast.Subscript)) else node.func
    return node.id if isinstance(node, ast.Name) else None
#===========================================================================
#               Validate the Code
#===========================================================================

SCALAR_METHODS = {"mean", "sum", "min", "max", "median", "std", "var", "count", "nunique"}

def _final_method_name(node: ast.AST) -> str | None:
    """اسم آخر method مستدعى بنهاية سلسلة التعبير (اللي بيحدد شكل الناتج النهائي)."""
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
        return node.func.attr
    return None

def validate_code(code: str) -> tuple[bool, str]:
    text_upper = code.upper()
    for kw in FORBIDDEN_SQL:
        if re.search(rf"\b{kw}\b", text_upper):
            return False, f"كلمة SQL ممنوعة: {kw}"

    try:
        tree = ast.parse(code, mode="exec")
    except SyntaxError as e:
        return False, f"صيغة كود غير صالحة: {e}"

    print_arg = None
    for node in ast.walk(tree):
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Name)
                and node.func.id == "print" and node.args):
            print_arg = node.args[0]
            break

    if print_arg is None or _final_method_name(print_arg) not in SCALAR_METHODS:
        return False, (
            "مسموح فقط بعمليات ترجع رقم واحد: " + ", ".join(sorted(SCALAR_METHODS))
            )

    for node in ast.walk(tree):
        if isinstance(node, FORBIDDEN_NODE_TYPES):
            return False, f"بنية كود غير مسموحة: {type(node).__name__}"

        if isinstance(node, ast.Attribute) and node.attr in FORBIDDEN_ATTRS:
            return False, f"وصول ممنوع لخاصية: {node.attr}"

        if isinstance(node, ast.Name) and node.id in FORBIDDEN_MODULES:
            return False, f"استخدام ممنوع لاسم: {node.id}"

        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, (ast.Subscript, ast.Attribute)):
                    root = _get_root_name(target)
                    if root in ("df", "pd"):
                        return False, f"تعديل مباشر على {root} غير مسموح"

        if isinstance(node, ast.Call):
            func = node.func
            if isinstance(func, ast.Name):
                if func.id in FORBIDDEN_CALLS:
                    return False, f"استدعاء دالة ممنوعة: {func.id}"
                if func.id not in SAFE_BUILTIN_CALLS:
                    return False, f"دالة غير مصرح بها: {func.id}"
            elif isinstance(func, ast.Attribute):
                root = _get_root_name(func)
                if root not in ("df", "pd"):
                    return False, f"استدعاء method على متغير غير مسموح: {root}"

    return True, "OK"

# ══════════════════════════════════════════════════════════
#  الجزء 5: Parsing رد الموديل
# ══════════════════════════════════════════════════════════

def _extract_code_line(text: str) -> str | None:
    for line in text.splitlines():
        line = line.strip().strip("`")
        if not line:
            continue
        if line.startswith("print("):
            m = re.search(r"print\(.+\)", line)
            if m:
                return m.group(0)
        elif line.startswith("df[") or line.startswith("df."):
            return f"print({line})"
    return None
#===========================================================================================================
#                                     تعديلات الدالة والاسباب
#===========================================================================================================
    #المحاولة 1 و2: هلق رح تلقط [CREATE_TICKET]: print(df['age'].min()) المدفون جوّا الرد فورًا بأول دورة،
    #بدل ما تدخل بحوار ASK_MORE_INFORMATION غير ضروري.
    #المحاولة 3: نفس السلوك الحالي (أول CREATE_TICKET بكود صالح) — ما تغيّر شي.
    #كبونص: لو فعلاً صار ASK_MORE_INFORMATION حقيقي (ما فيه كود جاهز بالرد أصلًا)، هلق منقص الرد عند أي تاغ زايد
    #جواه — فما رح تشوف تاغات خام متل [CREATE_TICKET]: مطبوعة بمنتصف رسالة الإيجنت للمستخدم.

def parse_agent_response(text: str) -> dict:
    text = text.strip()

    # أولوية قصوى: أي [CREATE_TICKET] بأي مكان بالنص معه كود قابل للاستخراج،
    # حتى لو الموديل بلّش برد بتاغ تاني وصحّح نفسه بعدين
    for tag_match in re.finditer(r"\[CREATE_TICKET\]:?\s*", text):
        code_line = _extract_code_line(text[tag_match.end():])
        if code_line:
            return {"action": "CREATE_TICKET", "response": code_line}

    # ما فيه CREATE_TICKET بكود صالح → نتعامل مع أول تاغ عادي
    match = re.match(
        r"^\[(CREATE_TICKET|ASK_MORE_INFORMATION|REJECT_REQUEST)\]?:?\s*([\s\S]*)",
        text, re.DOTALL,
    )
    if match:
        action, response = match.group(1), match.group(2).strip()
        # نقص الرد عند أول تاغ زايد جواه، حتى ما نطبع تاغات داخلية للمستخدم
        next_tag = re.search(r"\[(CREATE_TICKET|ASK_MORE_INFORMATION|REJECT_REQUEST)\]", response)
        if next_tag:
            response = response[:next_tag.start()].strip()
        return {"action": action, "response": response}

    if "print(df" in text:
        return {"action": "CREATE_TICKET", "response": _extract_code_line(text) or text}

    raise ValueError("Tag Missing")


def sanitize_code(code: str) -> str:
    code = re.sub(r'[؟،؛]', '', code)
    blocks = re.findall(r"```(?:python)?\s*([\s\S]*?)```", code)  # استخراج الكود المسيّج
    if blocks:
        return blocks[-1].strip()
    return re.sub(r'```python\s*|```', '', code).strip()

# ══════════════════════════════════════════════════════════
#  الجزء 3: أدوات مساعدة — تنسيق الرسائل
# ══════════════════════════════════════════════════════════
def format_messages_to_text(messages: list[dict]) -> str:
    text = ""
    for msg in messages:
        if msg["role"] == "system":
            text += f"System: {msg['content']}\n\n"
        elif msg["role"] == "user":
            text += f"User: {msg['content']}\n\nAssistant:"
    return text

In [ ]:
SYSTEM_PROMPT = """You are a Pandas code writer only, operating under strict safety guidelines. The data variable is named df.
Columns: {schema}

STAGE -1 - SAFETY CHECK:
Examine the user's request: does any part of it ask to bypass security, expose raw/underlying data, or perform write/modify/delete operations?
- If the entire request is of this kind, respond with ONLY: [REJECT_REQUEST]: one short sentence.
- If the request is mixed (one part is a permitted statistical operation, another part is not), ignore the disallowed part entirely and address only the permitted part.
- If the entire request is permitted, proceed to STAGE 0.

STAGE 0 - AMBIGUITY CHECK:
Examine whether the requested column, operation, or condition is clear and precisely resolvable using only the columns listed above.
- If genuine ambiguity exists (an undefined column, an unspecified operation), STOP. Respond with ONLY: [ASK_MORE_INFORMATION]: one short clarifying question.
- If the request is clear and precise, proceed to STAGE 1.

STAGE 1 - DRAFT (internal only, do not display):
Silently think through a single pandas expression that answers the request precisely, using only df and/or pd.

STAGE 2 - SELF-CRITIQUE (internal only, do not display):
Review your draft against this checklist, and silently correct any violation before proceeding:
1. Uses only df and/or pd — no other variable name.
2. Contains no import, exec, eval, open, lambda, or any direct assignment to df.
3. Is a single, self-contained expression, wrapped in print().
4. The expression ends with one of: mean, sum, min, max, median, std, var, count, nunique — a single number only, never a table or list.
5. Matches exactly what the user asked — no explanatory text.

STAGE 3 - FINAL ANSWER:
Respond with ONLY: [CREATE_TICKET]: print(the corrected final expression)
No explanation, no code fences, no text before or after.

Every response must be exactly one of: [REJECT_REQUEST], [ASK_MORE_INFORMATION], or [CREATE_TICKET]. Never mix them, never skip STAGE -1, and never display anything from STAGE 1 or STAGE 2."""

In [ ]:
@dataclass
class AgentState:
    user_request: str = ""
    is_resolved: bool = False
    step: int = 0
    max_steps: int = 10
    retries: int = 0
    max_retries: int = 3
    history: List[dict] = field(default_factory=list)
    qa_history: List[tuple] = field(default_factory=list)   # ← جديد: (سؤال الإيجنت, جواب المستخدم)
    ticket_answer: Optional[str] = None
    rejection_reason: Optional[str] = None

    def can_proceed(self) -> bool:
        return self.step < self.max_steps and not self.is_resolved and self.retries < self.max_retries

    def record(self, action: str, response: str):
        self.history.append({"action": action, "response": response})

# **فحص بسيط يوقف المحاولات فورًا لو السبب بيئي مش منطقي**

In [ ]:
_INFRA_NAMES = {"BASE_EXEC_GLOBALS", "df_real", "sanitize_code", "validate_code", "pd"}

def _is_environment_error(reason: str) -> bool:
    """خطأ بيئة تشغيل (متغير أساسي مفقود) — إعادة توليد الكود ما رح تصلحه أبدًا."""
    return "is not defined" in reason and any(f"'{name}'" in reason for name in _INFRA_NAMES)

# **تقييد نطاق التنفيذ**

In [ ]:
BASE_EXEC_GLOBALS = {
    "pd": pd,
    "__builtins__": {
        "print": print, "round": round, "len": len, "sum": sum,
        "min": min, "max": max, "abs": abs, "int": int, "float": float,
        "str": str, "list": list, "dict": dict, "sorted": sorted, "set": set,
    },
}

In [ ]:
def _extract_expr_from_print(code: str) -> str | None:
    """Extracts the inner expression from a print() call."""
    m = re.search(r"print\((.+)\)", code, re.DOTALL)
    if m:
        return m.group(1).strip()
    return None


In [ ]:
#==========================================================================
#             Self-Correct Prompt and helper Func
#==========================================================================

CORRECTION_PROMPT = """أنت تصلح كود Pandas معطوب. متغير البيانات اسمه df.
الأعمدة: {schema}

الكود السابق:
{previous_code}

سبب الفشل:
{failure_reason}

قيد إلزامي: التعبير النهائي لازم ينتهي بدالة ترجع رقمًا واحدًا فقط
(mean, sum, min, max, median, std, var, count, nunique) — ممنوع أي جدول
أو قيمة لكل مجموعة. لو الطلب أصلًا يحتاج تفصيل لكل فئة، اختار أقرب
ملخص عام رقمي واحد يعبّر عنه (مثال: بدل قيمة لكل فئة، اسنخدم std()
لقياس مدى التفاوت العام).

أعيد كتابة تعبير pandas صحيح ضمن هالقيد، بصيغة: print(التعبير)
سطر واحد فقط. بدون أي شرح، بدون أسوار كود، بدون أي نص قبله أو بعده."""


def _run_pandas_code(code: str, df_real: pd.DataFrame) -> tuple[bool, str]:
    expr = _extract_expr_from_print(code)
    if expr is None:
        return False, "لم يتم العثور على تعبير print() صالح"

    exec_globals = dict(BASE_EXEC_GLOBALS)
    exec_globals["df"] = df_real.copy()

    try:
        value = eval(expr, exec_globals)
    except Exception as e:
        return False, str(e)

    if not pd.api.types.is_scalar(value):
        return False, "الناتج ليس رقمًا واحدًا — العملية أنتجت مجموعة أو جدول، غير مسموح"

    result = str(value).strip()
    if not result:
        return False, "الناتج فارغ"

    print(result)
    return True, result

def _is_policy_violation(reason: str) -> bool:
    """رفض ناجم عن قيد التصميم (ناتج غير-scalar)، مش خطأ بمنطق الكود —
    إعادة توليد نفس الطلب لن يغيّره من جدول لرقم واحد."""
    return reason.startswith("مسموح فقط بعمليات ترجع رقم واحد") or reason.startswith("الناتج ليس رقمًا واحدًا")


def self_correct(code: str, failure_reason: str, schema: str, pipe, df_real, max_attempts: int = 3):
    """
    يحاول تصحيح كود فشل (أمنيًا أو تنفيذيًا)، عبر برومبت منفصل مخصص فقط
    للتصحيح. صامت أثناء المحاولات (بدون شرح ولا رسائل خطأ) — بيطبع بس
    رقم المحاولة الحالية، وبيطبع النتيجة النهائية فقط لو نجح.
    """
    current_code, current_reason = code, failure_reason

    for attempt in range(1, max_attempts + 1):
        print(f"🔄 محاولة تصحيح {attempt}/{max_attempts}")

        prompt = CORRECTION_PROMPT.format(schema=schema, previous_code=current_code, failure_reason=current_reason)
        messages = [{"role": "system", "content": prompt}, {"role": "user", "content": "صححي الكود."}]

        try:
            raw = generate_response(format_messages_to_text(messages), pipe)
        except Exception:
            current_reason = "فشل توليد رد من الموديل"
            continue

        candidate = _extract_code_line(raw)
        if candidate is None:
            current_code, current_reason = raw, "لم يُستخرج كود صالح من الرد"
            continue

        is_valid, reason = validate_code(candidate)
        if not is_valid:
            if _is_policy_violation(reason):          # ← جديد
                return False, None, None
            current_code, current_reason = candidate, reason
            continue

        success, result = _run_pandas_code(candidate, df_real)
        if not success:
            if _is_environment_error(result) or _is_policy_violation(result):   # ← وسّعنا الشرط
                return False, None, None
            current_code, current_reason = candidate, result
            continue

        print(result)
        return True, candidate, result

    return False, None, None

In [ ]:
#=============================================================================
#                      [Decision Graph]
#=============================================================================
import itertools
import networkx as nx  # مكتبة رسم بياني قياسية، موجودة افتراضيًا بـ Colab


class DecisionGraph:
    """KG بسيط: شو شاف الموديل، شو الخيارات، شو اختار، شو استبعد، ووين وصل ولماذا.
    كل المعلومات مبنية من مخرجات pipeline الموجود، بدون طلب تفسير جديد من الموديل."""

    ALL_OPTIONS = ("CREATE_TICKET", "ASK_MORE_INFORMATION", "REJECT_REQUEST")

    def __init__(self, session_id: str, schema: str):
        self.g = nx.DiGraph()
        self.session_id = session_id
        self._counter = itertools.count(1)
        self.g.add_node(session_id, type="session")
        self.g.add_node("schema", type="context", text=schema)
        for opt in self.ALL_OPTIONS:
            self.g.add_node(f"option::{opt}", type="option")

    def log_turn(self, user_input: str, chosen: str | None, reason: str, outcome: str) -> str:
        turn_id = f"turn_{next(self._counter)}"
        self.g.add_node(turn_id, type="turn", input=user_input)
        self.g.add_edge(self.session_id, turn_id, relation="HAS_TURN")
        self.g.add_edge(turn_id, "schema", relation="SAW_CONTEXT")

        for opt in self.ALL_OPTIONS:
            self.g.add_edge(turn_id, f"option::{opt}", relation="CONSIDERED")

        if chosen:
            self.g.add_edge(turn_id, f"option::{chosen}", relation="CHOSE")
            for opt in self.ALL_OPTIONS:
                if opt != chosen:
                    self.g.add_edge(turn_id, f"option::{opt}", relation="EXCLUDED")

        outcome_id = f"{turn_id}_outcome"
        self.g.add_node(outcome_id, type="outcome", label=outcome, reason=reason)
        self.g.add_edge(turn_id, outcome_id, relation="RESULTED_IN")
        return turn_id

    def print_trace(self):
        print("=" * 50)
        print("🧠 خريطة القرار")
        print("=" * 50)
        for node, data in self.g.nodes(data=True):
            if data.get("type") != "turn":
                continue
            chosen = [v.split("::")[1] for _, v, d in self.g.out_edges(node, data=True) if d["relation"] == "CHOSE"]
            excluded = [v.split("::")[1] for _, v, d in self.g.out_edges(node, data=True) if d["relation"] == "EXCLUDED"]
            outcome_node = [v for _, v, d in self.g.out_edges(node, data=True) if d["relation"] == "RESULTED_IN"][0]
            outcome = self.g.nodes[outcome_node]

            print(f"\n🔹 {node}")
            print(f"   الطلب: {data['input']}")
            print(f"   الخيارات المتاحة: {', '.join(self.ALL_OPTIONS)}")
            print(f"   اختار: {chosen[0] if chosen else 'لا شيء (فشل التحليل)'}")
            if excluded:
                print(f"   استبعد: {', '.join(excluded)}")
            print(f"   النتيجة: {outcome['label']}")
            print(f"   السبب: {outcome['reason']}")
        print("=" * 50)

    def export_graphml(self, path: str = None):
        path = path or f"{self.session_id}.graphml"
        nx.write_graphml(self.g, path)
        return path

# **Runner**

In [ ]:

# ══════════════════════════════════════════════════════════
#  الجزء 6: استدعاء الموديل
# ══════════════════════════════════════════════════════════

_DEGENERATE_PATTERN = re.compile(r'(.)\1{7,}')  # نفس الحرف مكرر 8 مرات متتالية فأكتر = هلوسة تكرارية

def generate_response(prompt_text: str, pipe) -> str:
    if pipe is None:
        raise RuntimeError("llm_pipe غير معرّف. شغّل خلية تحميل الموديل أولاً.")

    output = pipe(
        prompt_text,
        max_new_tokens=300,
        return_full_text=False,
        #repetition_penalty=1.3,     # ← جديد: يعاقب تكرار نفس التوكن
        #no_repeat_ngram_size=4,     # ← جديد: يمنع تكرار نفس تتابع 4 توكنات حرفيًا
    )
    generated = output[0]["generated_text"]

    for marker in ("\nUser:", "\nSystem:", "\nAssistant:"):
        idx = generated.find(marker)
        if idx != -1:
            generated = generated[:idx]
    generated = generated.strip()

    if _DEGENERATE_PATTERN.search(generated):          # ← جديد
        raise ValueError("رد متكرر/متدهور من الموديل")

    return generated
# ══════════════════════════════════════════════════════════
#  الجزء 7: نطاق التنفيذ المقيّد + الحلقة الرئيسية
# ══════════════════════════════════════════════════════════
def run_agent(user_request: str, pipe):
    state = AgentState()
    graph = DecisionGraph(session_id=f"session_{id(state)}", schema=DATA_SCHEMA)   # ← جديد
    current_input = user_request
    print(f"🧑 المستخدم: {current_input}\n")

    while state.can_proceed():
        sys_prompt = SYSTEM_PROMPT.format(schema=DATA_SCHEMA)
        user_text = (
            f"طلب المستخدم: {current_input}" if not state.history
            else f"السياق:\n{chr(10).join(h['response'] for h in state.history)}\n\nطلب المستخدم الجديد: {current_input}"
        )

        try:
            messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_text}]
            raw = generate_response(format_messages_to_text(messages), pipe)
            parsed = parse_agent_response(raw)
        except Exception as e:
            state.retries += 1
            graph.log_turn(current_input, chosen=None, reason=str(e), outcome="PARSE_FAILED")   # ← جديد
            print(f"⚠️ خطأ في صيغة الرد ({state.retries}/{state.max_retries}): {e}\n")
            if state.retries >= state.max_retries:
                state.rejection_reason = "فشل في صياغة رد صالح."
                state.is_resolved = True
            continue

        action, response = parsed["action"], parsed["response"]

        if action == "ASK_MORE_INFORMATION":
            state.record(action, response)
            graph.log_turn(current_input, chosen=action, reason=response, outcome="ASKED")
            state.retries = 0
            print(f"🤖 الإيجنت: {response}\n")
            reply = input("🧑 ردك: ")
            state.qa_history.append((response, reply))   # ← جديد: نحفظ السؤال مع جوابه الفعلي مع بعض
            current_input = reply

        elif action == "REJECT_REQUEST":
            state.rejection_reason = response
            state.is_resolved = True
            state.record(action, response)
            graph.log_turn(current_input, chosen=action, reason=response, outcome="REJECTED")   # ← جديد
            print(f"🛑 الإيجنت [مرفوض]: {response}\n")

        elif action == "CREATE_TICKET":
            is_valid, reason = validate_code(response)
            ok, result = _run_pandas_code(response, df_real) if is_valid else (False, reason)

            if ok:
                state.ticket_answer = result
                state.is_resolved = True
                state.step += 1
                state.retries = 0
                state.record(action, f"النتيجة: {result}\nالكود: {response}")
                graph.log_turn(current_input, chosen=action, reason=f"الكود: {response} → الناتج: {result}", outcome="SUCCESS")
                print(f"✅ النتيجة:\n{result}\n💻 الكود:\n{response}\n")
            else:
                if _is_environment_error(result):
                    state.rejection_reason = f"خطأ بيئة تشغيل: {result} — تأكد من تشغيل كل خلايا الإعداد بالترتيب."
                    state.is_resolved = True
                    state.record(action, "[ENVIRONMENT ERROR]")
                    graph.log_turn(current_input, chosen=action, reason=result, outcome="ENV_ERROR")
                else:
                    success, fixed_code, fixed_result = self_correct(response, result, DATA_SCHEMA, pipe, df_real)
                    if success:
                        state.ticket_answer = fixed_result
                        state.is_resolved = True
                        state.step += 1
                        state.record(action, f"[SELF-CORRECTED] {fixed_code} → {fixed_result}")
                        graph.log_turn(current_input, chosen=action,
                                        reason=f"تصحيح ذاتي بعد فشل ({result}) → {fixed_code} → {fixed_result}", outcome="SUCCESS")
                    else:
                        state.rejection_reason = f"فشل التصحيح الذاتي بعد 3 محاولات (السبب الأصلي: {result})"
                        state.is_resolved = True
                        state.record(action, "[SELF-CORRECTION FAILED]")
                        graph.log_turn(current_input, chosen=action, reason=result, outcome="ERROR")
            continue

    print("=" * 50)
    if state.ticket_answer:
        print("🏁 نجاح!")
    elif state.rejection_reason:
        print("⛔ رفض/فشل.")
    else:
        print("⏱️ انتهت المحاولات.")
    print("=" * 50)

    graph.print_trace()          # ← جديد
    return state, graph          # ← جديد: رجّعنا الغراف مع الحالة

In [ ]:
# Check dependencies
for name in ("_extract_expr_from_print", "_run_pandas_code", "BASE_EXEC_GLOBALS", "validate_code"):
    print(name, "→", "موجودة ✅" if name in globals() else "مفقودة ❌")

#run_agent("how does revenue vary across salary over time, and include one concrete occurence per salary so the pattern feels tangible", llm_pipe)
run_agent("in one-shot ,count the revenue over all salary column and its variation over years of experience column", llm_pipe)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


_extract_expr_from_print → موجودة ✅
_run_pandas_code → موجودة ✅
BASE_EXEC_GLOBALS → موجودة ✅
validate_code → موجودة ✅
🧑 المستخدم: in one-shot ,count the revenue over all salary column and its variation over years of experience column

🤖 الإيجنت: Could you please specify if the column 'revenue' exists in the dataframe df, and if so, how it is represented in the data?

🧑 ردك: it is not exsist , you should detect it by yourself


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 الإيجنت: هل يمكنني التحقق من موجود عمود 'الإيرادات' في فقرة df؟

🧑 ردك: عمود الراتب هو نفسه الايرادات


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 الإيجنت: هل 'الراتب' يشير إلى عمود 'الإيرادات' في القطاع البياني الخاص بي؟

🧑 ردك: نعم


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔄 محاولة تصحيح 1/3
1.7888543819998317
1.7888543819998317
🏁 نجاح!
🧠 خريطة القرار

🔹 turn_1
   الطلب: in one-shot ,count the revenue over all salary column and its variation over years of experience column
   الخيارات المتاحة: CREATE_TICKET, ASK_MORE_INFORMATION, REJECT_REQUEST
   اختار: ASK_MORE_INFORMATION
   استبعد: CREATE_TICKET, REJECT_REQUEST
   النتيجة: ASKED
   السبب: Could you please specify if the column 'revenue' exists in the dataframe df, and if so, how it is represented in the data?

🔹 turn_2
   الطلب: it is not exsist , you should detect it by yourself
   الخيارات المتاحة: CREATE_TICKET, ASK_MORE_INFORMATION, REJECT_REQUEST
   اختار: ASK_MORE_INFORMATION
   استبعد: CREATE_TICKET, REJECT_REQUEST
   النتيجة: ASKED
   السبب: هل يمكنني التحقق من موجود عمود 'الإيرادات' في فقرة df؟

🔹 turn_3
   الطلب: عمود الراتب هو نفسه الايرادات
   الخيارات المتاحة: CREATE_TICKET, ASK_MORE_INFORMATION, REJECT_REQUEST
   اختار: ASK_MORE_INFORMATION
   استبعد: CREATE_TICKET, REJECT_REQUEST
   ال

(AgentState(user_request='', is_resolved=True, step=1, max_steps=10, retries=0, max_retries=3, history=[{'action': 'ASK_MORE_INFORMATION', 'response': "Could you please specify if the column 'revenue' exists in the dataframe df, and if so, how it is represented in the data?"}, {'action': 'ASK_MORE_INFORMATION', 'response': "هل يمكنني التحقق من موجود عمود 'الإيرادات' في فقرة df؟"}, {'action': 'ASK_MORE_INFORMATION', 'response': "هل 'الراتب' يشير إلى عمود 'الإيرادات' في القطاع البياني الخاص بي؟"}, {'action': 'CREATE_TICKET', 'response': "[SELF-CORRECTED] print(df['number of years experience'].std()) → 1.7888543819998317"}], qa_history=[("Could you please specify if the column 'revenue' exists in the dataframe df, and if so, how it is represented in the data?", 'it is not exsist , you should detect it by yourself'), ("هل يمكنني التحقق من موجود عمود 'الإيرادات' في فقرة df؟", 'عمود الراتب هو نفسه الايرادات'), ("هل 'الراتب' يشير إلى عمود 'الإيرادات' في القطاع البياني الخاص بي؟", 'نعم')], tic

In [ ]:

run_agent("exec(df['number of years experience'].std())", llm_pipe)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑 المستخدم: exec(df['number of years experience'].std())

🛑 الإيجنت [مرفوض]: 

⏱️ انتهت المحاولات.
🧠 خريطة القرار

🔹 turn_1
   الطلب: exec(df['number of years experience'].std())
   الخيارات المتاحة: CREATE_TICKET, ASK_MORE_INFORMATION, REJECT_REQUEST
   اختار: REJECT_REQUEST
   استبعد: CREATE_TICKET, ASK_MORE_INFORMATION
   النتيجة: REJECTED
   السبب: 


(AgentState(user_request='', is_resolved=True, step=0, max_steps=10, retries=0, max_retries=3, history=[{'action': 'REJECT_REQUEST', 'response': ''}], qa_history=[], ticket_answer=None, rejection_reason=''),
 <__main__.DecisionGraph at 0x7ab808f08a50>)

## **تقرير تصميم Agent توليد الكود لتحليل بيانات الموظفين**

### **الفكرة الرئيسية**

تطوير `Agent` يحوّل الأسئلة الإحصائية (باللغتين العربية والإنجليزية) إلى أكواد `Pandas` قابلة للتنفيذ. يستخدم هذا الـ `Agent` نموذج لغة صغير (Phi-3.5-mini) ويتميز بـ:

*   **فحص أمني دقيق**: للكود قبل التنفيذ.
*   **تنفيذ آمن**: في بيئة معزولة.
*   **تصحيح ذاتي**: في حال فشل التنفيذ الأولي.
*   **تسجيل القرار**: كل خطوة تُسجّل في `Knowledge Graph`.

> **المبدأ الحاكم**: `الموديل صغير وضعيف الالتزام بالتعليمات ← الكود هو المسؤول عن الانضباط، وليس البرومبت وحده.`

### **الطبقات المعمارية**

| الطبقة          | الوظيفة الأساسية                                                            |
| :-------------- | :-------------------------------------------------------------------------- |
| **البيانات**      | الموديل يرى **أسماء الأعمدة فقط**، وليس القيم الفعلية.                   |
| **الأمان**        | **فحص AST (Deny-by-Default)**: يمنع الاستدعاءات والبنى غير المصرح بها. |
| **التنفيذ**       | كل عملية تنفيذ تتم على **نسخة معزولة (`df.copy()`)** لضمان عدم التأثير على البيانات الأصلية. |
| **الحوار**        | استخدام **برومبت متعدد المراحل (5 مراحل)**، مع تفكير ونقد ذاتي داخلي.       |
| **التصحيح الذاتي** | **برومبت مستقل ومحدد (3 محاولات)**، يعمل بصمت تام.                       |
| **الملاحظة**      | **خريطة القرار (`DecisionGraph`)**: تسجل الطلب، الخيارات، المختار، المستبعد، والسبب. |

### **مسار معالجة السؤال (خطوة بخطوة)**

1.  **بناء البرومبت**: يشمل القواعد، أسماء الأعمدة، وسياق المحاولات السابقة.
2.  **توليد رد نظيف**: من النموذج، مع إزالة تكرار البرومبت وأي هلوسة محادثة.
3.  **تحديد القرار**: الأولوية لأي كود `Pandas` صحيح يتم العثور عليه في الرد، حتى لو كان النموذج قد بدأ برد مختلف.
4.  **معالجة 'الكود'**: إذا كان القرار يتضمن كوداً:
    *   **فحص أمني**: يليه تنفيذ معزول.
    *   **نجاح التنفيذ؟**: تُعرض النتيجة مباشرة.
    *   **فشل التنفيذ؟**: يُفعّل التصحيح الذاتي (3 محاولات صامتة).
    *   **استثناء**: إذا كان الفشل بسبب `خطأ بيئة تشغيل` (مثل متغير غير معرّف)، يتوقف التصحيح فوراً لعدم جدواه.
5.  **تسجيل وحفظ**: كل خطوة تُسجَّل في `خريطة القرار` وتُطبع في نهاية الجلسة.

### **القرارات التصميمية الرئيسية والأسباب الكامنة**

| القرار                                                | السبب الجوهري                                                                                                                                                             |
| :--------------------------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **رؤية الأعمدة فقط**                                   | **يقلل مساحة الخطر**: يمنع الموديل من تسريب بيانات حساسة ويفرض عليه التركيز على منطق الأعمدة.                                                                       |
| **فحص AST بدلاً من Regex**                            | **يفهم بنية الكود**: يوفر فحصًا أمنيًا حقيقيًا يصعب الالتفاف عليه، مما يجعله أكثر أمانًا من الفحص النصي.                                                              |
| **نسخة معزولة (`df.copy()`)**                         | **حماية البيانات**: توفر طبقة حماية مستقلة؛ حتى لو تجاوز كود ضار الفحص الأمني، لن يؤثر على البيانات الأصلية.                                                          |
| **مراحل التفكير داخلية**                                | **نص نظيف**: أي نص يسبق التاغ النهائي يكسر عملية استخلاص الرد؛ الحفاظ على هذه المراحل داخلية يضمن استخراجًا فعالاً.                                                     |
| **ضبط يدوي لمخرجات الموديل**                           | **تنظيف الردود**: الموديل الافتراضي يعيد البرومبت كاملاً ملتصقًا بالرد؛ القص اليدوي ضروري للحصول على ردود نظيفة وخالية من الهلوسة.                                  |
| **أولوية لأي كود صحيح**                                | **المرونة والفعالية**: الموديلات الصغيرة غالبًا ما "تفكر بصوت عالٍ" وتصحح نفسها؛ هذا يضمن التقاط الحل الصحيح حتى لو لم يكن في البداية.                               |
| **عداد المحاولات لفشل التحليل فقط**                     | **تجنب اللانهائية**: يمنع إعادة تعيين العداد بشكل غير مقصود، مما كان قد يلغي الحد الأقصى للمحاولات فعليًا ويؤدي إلى حلقات لا نهائية.                                |
| **التصحيح الذاتي المنفصل**                             | **فصل الاهتمامات**: يفصل مهمة "تصنيف الطلب" عن "تصليح الكود"، مما يقلل من هلوسة النموذج ويجعل برومبت التصحيح أكثر تركيزًا وفعالية.                               |
| **التصحيح الذاتي يخضع لنفس الفحص الأمني**               | **لا استثناءات أمنية**: يضمن أن الكود المُصحح يمر بنفس معايير الأمان والتنفيذ المعزول، فلا توجد ثغرات أمنية في عملية التصحيح.                                           |
| **كشف "خطأ البيئة" وإيقاف التصحيح فوراً**             | **توفير الموارد**: خطأ مثل "متغير غير معرّف" لا يمكن تصحيحه بإعادة توليد الكود؛ إيقاف التصحيح يوفر محاولات غير مجدية.                                               |
| **خريطة القرار مبنية على مخرجات موثوقة**                | **موثوقية المعلومات**: تضمن أن خريطة القرار تستند إلى حقائق ملموسة من الـ Pipeline، بدلاً من تفسيرات حرة من الموديل قد تكون عرضة للهلوسة.                            |


# *   **ملاحظة ضبط مهمة للملف**
    *   ملفك 5 صفوف → `print(df['salary'])` بيطلع 6 أسطر → عند حد 10 الفاليديشن ما بيمسكه.
    *   الدفاع الأساسي صار البرومبت (بند 4 بـ STAGE 2 — ممنوع بالاسم)
    *   الفاليديشن شبكة أمان للملفات الكبيرة (1000 صف رح يمسكها فوراً)
    *   لو بدك الفاليديشن يمسك حالة ملفك الصغير كمان: خلّي `MAX_OUTPUT_LINES = 5`
    *   وبالمناسبة المطمئنة: حتى لو الموديل ولّد `inplace=True` جوّا `print(...)` → بيطبع `None` → بيقع في `[NO OUTPUT]` → بيصحح نفسه، و`df_real.copy()` بتحمي بياناتك الأصلية مهما صار

## 🚀 ملخص تصميم Agent النظام

### طبقات الإيجنت (بترتيب التنفيذ)

*   **البيانات والـ Schema**: `df_real` يُقرأ مرة واحدة، `DATA_SCHEMA` (أسماء أعمدة فقط) يُحقن بالبرومبت.
*   **طبقة الأمان**: `validate_code` (فحص AST + allow-list) قبل أي تنفيذ.
*   **طبقة التنفيذ**: `_run_pandas_code`: تنفيذ معزول بنسخة `df.copy()`، يرجع (نجح؟, ناتج/سبب فشل).
*   **الرسائل والتحليل**: `format_messages_to_text`, `SYSTEM_PROMPT` (5 مراحل), `AgentState`, `parse_agent_response`.
*   **التوليد**: `generate_response`: `return_full_text=False` + قص أي هلوسة أدوار محادثة.
*   **التصحيح الذاتي**: `CORRECTION_PROMPT` + `self_correct`: حلقة مستقلة 3 محاولات.
*   **الملاحظة (Observability)**: `DecisionGraph`: يسجل كل قرار كعقد وعلاقات بـ `networkx`.
*   **الأوركستريشن**: `run_agent`: يربط كل الطبقات فوق ببعضها.

